# Experiment 3 — Residual Diagnostics

**PDF reference: Phase 3, Experiment 3**

After fitting the spline baseline, inspect the residuals for:
- **Residual vs avg_power** — should be a flat horizontal cloud (no power trend remaining)
- **Residual vs segment grade** — look for grade-speed confounding
- **Residual vs time colored by bike** — the level shift is the signal; within-bike residuals should be flat
- **QQ-plot** — residuals approximately normal → OLS standard errors are valid
- **KS power-overlap per segment** — flag segments where bikes rode at very different watts

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

from src.database import init_db, load_efforts, load_segments, load_bikes
from src.bike_delta import (
    prepare_delta_dataset, get_paired_segments,
    fit_baseline_model, compute_residuals,
    power_overlap_ok, segment_power_overlap_summary,
)
from src.analytics import filter_outliers_by_power_speed

sns.set_theme(style='whitegrid', palette='Set2')
print('Imports OK')

In [ ]:
# ── Load and prepare ──────────────────────────────────────────────────────────
init_db()
raw_efforts = load_efforts()
segments = load_segments()
bikes_df = load_bikes()
bikes_dict = dict(zip(bikes_df['gear_id'], bikes_df['name'])) if not bikes_df.empty else {}

power_efforts = raw_efforts[raw_efforts['average_watts'].notna()].copy()
df = prepare_delta_dataset(power_efforts, segments, bikes_dict)
df, _ = filter_outliers_by_power_speed(df, z_threshold=2.0)

all_bikes = df['bike_name'].dropna().unique().tolist()

# Configure
REF_BIKE = all_bikes[0]
SPLINE_DF = 5
MIN_EFFORTS = 3
KS_THRESHOLD = 0.05

print(f'Reference bike: {REF_BIKE}')
print(f'All bikes: {all_bikes}')

paired = get_paired_segments(df, all_bikes, min_efforts=MIN_EFFORTS)
model, design_info = fit_baseline_model(df, REF_BIKE, spline_df=SPLINE_DF)
df_resid = compute_residuals(df, model, design_info)
df_resid['_dt'] = pd.to_datetime(df_resid['start_date'], errors='coerce', utc=True).dt.tz_convert(None)

print(f'Residuals computed for {len(df_resid)} efforts')

## 3.1 — Residual vs avg_power
Should be a flat horizontal cloud.  A slope indicates the power term in the model did not fully capture power effects — consider a quadratic power term.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

palette = sns.color_palette('Set2', n_colors=len(all_bikes))
bike_color = dict(zip(all_bikes, palette))

for bike in all_bikes:
    bdata = df_resid[df_resid['bike_name'] == bike].dropna(subset=['residual', 'average_watts'])
    ax.scatter(bdata['average_watts'], bdata['residual'],
               label=bike, color=bike_color[bike], alpha=0.5, s=25)

# Lowess trend on all residuals
all_r = df_resid.dropna(subset=['residual', 'average_watts'])
from statsmodels.nonparametric.smoothers_lowess import lowess
lowess_fit = lowess(all_r['residual'], all_r['average_watts'], frac=0.3, return_sorted=True)
ax.plot(lowess_fit[:, 0], lowess_fit[:, 1], 'k-', lw=2, label='LOWESS trend')

ax.axhline(0, color='grey', linestyle='--', lw=1.5)
ax.set_xlabel('Avg power (W)')
ax.set_ylabel('Residual (speed/W¹⁄³)')
ax.set_title('Residual vs avg power — should be a flat horizontal cloud')
ax.legend()
plt.tight_layout()
plt.show()

# Correlation test
r_vals = all_r[['average_watts', 'residual']].dropna()
corr, pval = stats.pearsonr(r_vals['average_watts'], r_vals['residual'])
print(f'Pearson r(power, residual) = {corr:.3f}  p = {pval:.4f}')
if abs(corr) > 0.15:
    print('⚠️  Residuals correlated with power — model may not be correcting for power fully.')
else:
    print('✅  No strong power–residual correlation.')

## 3.2 — Residual vs segment grade
Grade-specific aerodynamic or gravitational effects show up here. A slope indicates grade is a confound.

In [ ]:
if 'average_grade' in df_resid.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    for bike in all_bikes:
        bdata = df_resid[df_resid['bike_name'] == bike].dropna(subset=['residual', 'average_grade'])
        ax.scatter(bdata['average_grade'], bdata['residual'],
                   label=bike, color=bike_color[bike], alpha=0.5, s=25)
    ax.axhline(0, color='grey', linestyle='--', lw=1.5)
    ax.set_xlabel('Segment average grade (%)')
    ax.set_ylabel('Residual (speed/W¹⁄³)')
    ax.set_title('Residual vs segment grade')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    r_vals = df_resid[['average_grade', 'residual']].dropna()
    corr_g, pval_g = stats.pearsonr(r_vals['average_grade'], r_vals['residual'])
    print(f'Pearson r(grade, residual) = {corr_g:.3f}  p = {pval_g:.4f}')
else:
    print('No average_grade column — cannot check grade confounding.')

## 3.3 — Residual vs time (key visual)
The level shift between bikes is the signal.  Within-bike residuals should be flat.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for bike in all_bikes:
    bdata = df_resid[df_resid['bike_name'] == bike].dropna(subset=['residual']).sort_values('_dt')
    ax.scatter(bdata['_dt'], bdata['residual'],
               label=bike, color=bike_color[bike], alpha=0.55, s=30)
    roll = bdata.set_index('_dt')['residual'].rolling('30D', min_periods=3).mean()
    ax.plot(roll.index, roll.values, color=bike_color[bike], lw=2.5, linestyle='--')

ax.axhline(0, color='grey', linestyle='--', lw=1.5)
ax.set_xlabel('Date')
ax.set_ylabel('Residual (speed/W¹⁄³)')
ax.set_title('Residuals vs time — the level shift between bikes is the signal')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 3.4 — QQ-plot (normality check)
OLS CIs rely on residuals being approximately normal.

In [ ]:
residuals_clean = df_resid['residual'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# QQ-plot
(osm, osr), (slope, intercept, r) = stats.probplot(residuals_clean, dist='norm')
axes[0].scatter(osm, osr, alpha=0.5, s=15)
line_x = np.array([osm[0], osm[-1]])
axes[0].plot(line_x, slope * line_x + intercept, 'r-', lw=2)
axes[0].set_xlabel('Theoretical quantiles')
axes[0].set_ylabel('Sample quantiles')
axes[0].set_title(f'QQ-plot of residuals  (R²={r**2:.3f})')

# Histogram with normal overlay
axes[1].hist(residuals_clean, bins=40, density=True, alpha=0.6, color='steelblue')
x = np.linspace(residuals_clean.min(), residuals_clean.max(), 200)
axes[1].plot(x, stats.norm.pdf(x, residuals_clean.mean(), residuals_clean.std()),
             'r-', lw=2, label='Normal fit')
axes[1].set_title('Residual distribution')
axes[1].set_xlabel('Residual')
axes[1].legend()

plt.tight_layout()
plt.show()

# Shapiro-Wilk (only valid for n < 5000)
sample = residuals_clean.sample(min(3000, len(residuals_clean)), random_state=42)
stat, p_sw = stats.shapiro(sample)
print(f'Shapiro-Wilk: W={stat:.4f}  p={p_sw:.4f}')
if p_sw < 0.05:
    print('⚠️  Residuals may not be normal — OLS CIs are approximate. Consider robust SEs.')
else:
    print('✅  Cannot reject normality of residuals.')

## 3.5 — KS power-overlap per segment

In [ ]:
ks_summary = segment_power_overlap_summary(
    df_resid, all_bikes, paired, p_threshold=KS_THRESHOLD
)

if ks_summary.empty:
    print('No KS results — check paired segments and power data.')
else:
    seg_names = dict(zip(segments['segment_id'], segments['name'])) \
        if 'name' in segments.columns else {}
    ks_summary['segment_name'] = ks_summary['segment_id'].map(lambda s: seg_names.get(s, str(s)))
    ks_summary['status'] = ks_summary['ks_ok'].map({True: '✅ OK', False: '⚠️ Flagged'})
    
    flagged = ks_summary[~ks_summary['ks_ok']]
    print(f'Flagged segments (KS p < {KS_THRESHOLD}): {len(flagged)} / {len(ks_summary)}')
    print()
    print(ks_summary[['segment_name', 'bike_a', 'bike_b', 'p_value', 'status']].to_string(index=False))